In [6]:
import os, sys, shutil
from glob import glob


GARMENT_VERSION = "02"

src_gmt_dir_list = sorted(glob(os.path.join(
    "gcd_01", "GCD_01", "garments_5000_1", "*"
)))



In [9]:

for src_gmt_dir in src_gmt_dir_list :
    tgt_gmt_dir = os.path.join(
        "SAMPLE_DATA", f"GCD__{GARMENT_VERSION}", os.path.basename(src_gmt_dir)
    )
    
    # chk_dir = os.path.join(
    #     "SAMPLE_DATA", f"GCD__{GARMENT_VERSION}__", os.path.basename(src_gmt_dir)
    # )
    # if os.path.exists(chk_dir) :continue
        

    os.makedirs(tgt_gmt_dir, exist_ok=True)
    
    # copy json files
    json_file_list = glob(os.path.join(src_gmt_dir, "*.json"))
    for json_file in json_file_list :
        shutil.copy(json_file, tgt_gmt_dir)
    
    # copy zprj files
    zprj_file_list = glob(os.path.join(src_gmt_dir, "*.zprj"))
    for zprj_file in zprj_file_list :
        shutil.copy(zprj_file, tgt_gmt_dir)
    

In [ ]:
# If garment_dir is not finished with GARMENT_VERSION, rename directory
dst_gmt_dir_list = sorted(glob(os.path.join(
    "SAMPLE_DATA", f"GCD__{GARMENT_VERSION}", "*"
)))

for dst_gmt_dir in dst_gmt_dir_list :
    if dst_gmt_dir[-2:] != GARMENT_VERSION :
        
        # print(dst_gmt_dir)
        # print(dst_gmt_dir + f"__{GARMENT_VERSION}")
        
        shutil.move(
            dst_gmt_dir,
            dst_gmt_dir + f"__{GARMENT_VERSION}"
        )


In [ ]:


import os, sys
import json
import time
from glob import glob
import random
import import_api
import utility_api
import fabric_api
import pattern_api
import export_api
import ApiTypes

from pathlib import Path
from dataclasses import dataclass

def get_sort_key(path_str):
    path = Path(path_str)
    # Find the 'garments_5000_X' part
    garment_dir = next(part for part in path.parts if part.startswith('garments_5000_'))
    # Extract the number after 'garments_5000_'
    garment_num = int(garment_dir.split('_')[-1])
    # Get the random ID from the path
    random_id = path.parent.name
    return (garment_num, random_id)

@dataclass
class PathConfig :
    root_path : str = None
    avatar_dir: str = "CLO_ASSETS/AVATARs"
    fabric_dir: str = "CLO_ASSETS/FABRICs"
    pose_dir: str = "CLO_ASSETS/POSEs"
    viewpoint_dir: str = "CLO_ASSETS/VIEWPOINTs"
    
    gcd_dir: str = "GarmentCodeData_v2"

    outfit_metadata_path: str = "gcd_01_outfit_path_list.json"
    combination_metadata_path: str = "gcd_01_top_bottom_path_list.json"
    
    sample_data_dir: str = r"gcd_01\GCD_01"
    
    outfit_path_list: list = None
    combination_path_list: list = None

    def __post_init__(self):
        self.avatar_path_list = sorted(glob(os.path.join(self.root_path, self.avatar_dir, "*.avt")))
        self.fabric_path_list = sorted(glob(os.path.join(self.root_path, self.fabric_dir, "*.zfab")))
        self.pose_path_list = sorted(glob(os.path.join(self.root_path, self.pose_dir, "*.pos")))
        self.viewpoint_path_list = sorted(glob(os.path.join(self.root_path, self.viewpoint_dir, "*.zcmr")))
        
        self.gcd_path_list = sorted(
            glob(os.path.join(
                self.root_path, self.gcd_dir,
                "*", "*", "*config.json"
            )),
            key=get_sort_key
        )
        
        self.sample_data_dir = os.path.join(self.root_path, self.sample_data_dir)
        
        self.outfit_metadata_path = os.path.join(self.root_path, self.outfit_metadata_path)
        self.combination_metadata_path = os.path.join(self.root_path, self.combination_metadata_path)

        with open(self.outfit_metadata_path, "r") as f:
            self.outfit_metadata = json.load(f)
        self.outfit_path_list = []
        for outfit in self.outfit_metadata:
            garment_split, _, garment_id = list(Path(outfit).parts)[-3:]
            
            self.outfit_path_list.append(os.path.join(
                self.sample_data_dir, garment_split, garment_id,
                f"{garment_id}__01__clo.json"
            ))

        with open(self.combination_metadata_path, "r") as f:
            self.combination_metadata_raw = json.load(f)
        self.combination_path_list = []
        for combination in self.combination_metadata_raw:
            top_base_path, bottom_base_path = combination.split(",")
            top_garment_split, _, top_garment_id = list(Path(top_base_path).parts)[-3:]
            bottom_garment_split, _, bottom_garment_id = list(Path(bottom_base_path).parts)[-3:]
            self.combination_path_list.append(os.path.join(
                self.sample_data_dir, top_garment_split,
                f"{top_garment_id}__01__{bottom_garment_id}__01",
                f"{top_garment_id}__01__{bottom_garment_id}__01__clo.json"
            ))
            
    @property
    def avatar_count(self) -> int:
        return len(self.avatar_path_list)

    @property
    def fabric_count(self) -> int:
        return len(self.fabric_path_list)
        
    @property
    def pose_count(self) -> int:
        return len(self.pose_path_list)
    
    @property
    def viewpoint_count(self) -> int:
        return len(self.viewpoint_path_list)

    @property
    def gcd_count(self) -> int:
        return len(self.gcd_path_list)
    


@dataclass
class GarmentScene :
    # is_combination: bool = False
    # garment_json_path: str = None
    garment_dir: str = None
    avatar_path: str = None
    whole_fabric_path_list: str = None
    fabric_path_list: list = None
    pose_path_list:list    = None
    panel_count_list: list = None
    # pose_path:str = None
    viewpoint_path_list: list = None
    GARMENT_VERSION: str = None
    
    def __post_init__(self):
        self.output_dir = self.garment_dir
        return
        
        # # identify if scene is composed of single or multiple garments
        # if len(os.path.basename(self.garment_json_path).split("__")) == 3:
        #     self.is_combination = False
        # elif len(os.path.basename(self.garment_json_path).split("__")) == 5:
        #     self.is_combination = True
        # else :
        #     raise ValueError(f"Invalid garment json path: {self.garment_json_path}")
        
        # self.panel_count_list, self.prev_fabric_count = self.get_panel_fabric_count_list()
        # self.fabric_path_list = list(map(
        #     lambda x : random.choice(self.whole_fabric_path_list),
        #     self.panel_count_list
        # ))
        # self.output_dir = os.path.dirname(self.garment_json_path)
        
    def get_panel_fabric_count_list(self) :
        with open(self.garment_json_path, "r") as f:
            json_data = json.load(f)
            
        panel_count_list = []
        cur_garment_id = None
        for pattern in json_data["PatternList"] :
            garment_id = pattern["ID"].split("_")[1]
            
            if garment_id == cur_garment_id :
                panel_count_list[-1] += 1
            else :
                cur_garment_id = garment_id
                panel_count_list.append(1)
            
        return panel_count_list, len(json_data["FabricList"])
             
    def delete_prev_output(self) :
        output_dir = self.garment_dir
        for file in os.listdir(output_dir):
            file_path = os.path.join(output_dir, file)
            if not file.endswith('.json') and os.path.isfile(file_path):
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(f"Error deleting {file_path}: {e}")
                    

    def import_garment_json(self, garment_json_path, fabric_path) :
        
        with open(garment_json_path, "r") as f :
            json_data = json.load(f)    
            panel_count = len(json_data["PatternList"])
        
        pattern_api.ImportPatternJSON(garment_json_path)
        fabric_api.AddFabric(fabric_path)
        
        # for fabric_path in self.fabric_path_list :
        #     fabric_api.AddFabric(fabric_path)
        
        fabric_idx = fabric_api.GetFabricCount(-2) - 1
        
        
        for panel_idx in range(panel_count) :
            fabric_api.AssignFabricToPattern(fabric_idx, panel_idx, fabric_idx)
            pattern_api.SetArrangementShapeStyle(panel_idx, "Flat")
        
        
    def main(
        self,
        option = None,
        zprj_option = None,
        SIM_STEP = 200,
        SIM_STEP_2 = 100,
        delete_prev_output = True,
        pass_when_zprj_exists = False,
    ) :
        if pass_when_zprj_exists :
            zprj_path = os.path.join(
                self.garment_dir,
                f"{os.path.basename(self.garment_dir)}.zprj"
            )
            if os.path.exists(zprj_path) :
                return
            
        if delete_prev_output : self.delete_prev_output()
        
        if zprj_option is None :
            zprj_option = ApiTypes.ImportZPRJOption()
            zprj_option.bAppend = True
            # zprj_option.bLoadAvatar = False
            # zprj_option.bLoadSceneAndProps = True
            # zprj_option.bLoadCustomView = True
        
        if option is None :
            option = ApiTypes.ImportExportOption()
            option.bExportGarment = True
            option.bExportAvatar  = True
            option.bSingleObject  = True
            option.bThin          = False
            option.bSaveInZip     = False
            option.bMetaData      = True
            
        utility_api.NewProject()
        import_api.ImportFile(self.avatar_path)    
        pattern_api.ImportPatternJSON(os.path.join(
            self.garment_dir, f"{os.path.basename(self.garment_dir)}__clo.json"
        ))
        export_api.ExportSnapshot3D(os.path.join(self.garment_dir, "pre_drape.png"))        


        garment_name_list = os.path.basename(self.garment_dir).split("__")[::2]
        garment_version_id = os.path.basename(self.garment_dir).split("__")[-1]
        
        garment_json_path_list = list(map(
            lambda x : os.path.join(self.garment_dir, f"{x}__{garment_version_id}__clo.json"),
            garment_name_list
        ))
        
        if len(garment_name_list) == 1 : # Only one garment in scene
            utility_api.NewProject()
            import_api.ImportFile(self.avatar_path)
            garment_json_path = garment_json_path_list[0]
            self.import_garment_json(garment_json_path, random.choice(self.whole_fabric_path_list))
            export_api.ExportZPrj(garment_json_path.replace(".json", ".zprj"))
            
            utility_api.Simulate(SIM_STEP)
        
        elif len(garment_name_list) == 2 : # Scene contains two garments. load each garment, and save to zprj file one by one, and load zprj files one by one
            
            for (garment_json_path, fabric_path) in zip(
                garment_json_path_list,
                [random.choice(self.whole_fabric_path_list), random.choice(self.whole_fabric_path_list)]
            ) :
                utility_api.NewProject()
                import_api.ImportFile(self.avatar_path)
            
                self.import_garment_json(garment_json_path, fabric_path)
                export_api.ExportZPrj(garment_json_path.replace(".json", ".zprj"))
        
            utility_api.NewProject()
            # for garment_json_path in reversed(garment_json_path_list) :
            #     import_api.ImportZprj(garment_json_path.replace(".json", ".zprj"), zprj_option)
            #     utility_api.Simulate(SIM_STEP)
            import_api.ImportZprj(garment_json_path_list[1].replace(".json", ".zprj"), zprj_option)
            utility_api.Simulate(SIM_STEP)
            utility_api.DeleteAvatar([0])
            import_api.ImportZprj(garment_json_path_list[0].replace(".json", ".zprj"), zprj_option)
            utility_api.Simulate(SIM_STEP)
            # utility_api.DeleteAvatar([0])
            # print(f"AVATAR COUNT : {export_api.GetAvatarCount()}")
            # print(export_api.GetAvatarNameList())
        
        else :
            raise ValueError(f"Invalid garment json path: {self.garment_dir}")
        
        zprj_path = os.path.join(
            self.garment_dir,
            f"{os.path.basename(self.garment_dir)}.zprj"
        )
        export_api.ExportZPrj(zprj_path)
        
        prev_pose_name = None 
        for viewpoint in self.viewpoint_path_list :
        
            utility_api.NewProject()
            import_api.ImportZprj(zprj_path, zprj_option)
            
            view_name = Path(viewpoint).stem
            import_api.ImportFile(viewpoint)
            
            pose_path = random.choice(self.pose_path_list)
            pose_name = Path(pose_path).stem
            
            import_api.ImportPose(pose_path)
            utility_api.Simulate(SIM_STEP_2)
            
            export_api.ExportOBJ(
                os.path.join(self.output_dir, f"{view_name}__{pose_name}.obj"),
                option
            )
            
            export_api.ExportRenderingImage(os.path.join(
                self.output_dir, f"{view_name}__{pose_name}.png"
            ))

        print(self.garment_dir)
            
        

SYSTEM_CONFIG_DICT = {
    "HJP_WINDOWS_DESKTOP": {
        # "CLO_DIR": r"E:\HJP\KUAICV\VTO\CLO_AUTO_GEN",
        "CLO_DIR": r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN"
    }
}
system_name = "HJP_WINDOWS_DESKTOP"

path_config = PathConfig(root_path=SYSTEM_CONFIG_DICT[system_name]["CLO_DIR"])

# garment_path_list = path_config.combination_path_list
# garment_path_list = path_config.outfit_path_list

GARMENT_VERSION = "02"
garment_dir_list = sorted(glob(os.path.join(
    r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA",
    f"GCD__{GARMENT_VERSION}",
    "*"
)))


for idx, garment_dir in enumerate(garment_dir_list) :
    garment_scene = GarmentScene(
        garment_dir=garment_dir,
        avatar_path=path_config.avatar_path_list[0],
        whole_fabric_path_list=path_config.fabric_path_list,
        viewpoint_path_list=path_config.viewpoint_path_list,
        pose_path_list=path_config.pose_path_list,
    )
    garment_scene.main(
        delete_prev_output=True,
        pass_when_zprj_exists=True
    )
    try :
        CLO_CACHE_DIR = r"C:\Users\HJP\AppData\Local\CLO Virtual Fashion\CLO Standalone OnlineAuth"
        shutil.rmtree(CLO_CACHE_DIR)
    except Exception as e :
        print(e)



In [13]:
fabric_path = r"E:\HJP\KUAICV\VTO\DATA\CLO\CLO_ASSETs\FABRICs"

print(
    len(glob(os.path.join(
        r"E:\HJP\KUAICV\VTO\DATA\CLO\CLO_ASSETs\FABRICs",
        r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\CLO_ASSETs\FABRICs",
        "*.zfab"
    )))
)

from glob import glob

glob(os.path.join(
    r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\sample_data", "*"
))

75


['D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_005CWF73E9__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_45V6ZGELXG__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_48R3G5YQID__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_4EI7RB9464__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_4JILJHEVIR__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\dress_sleeveless_4R3PV1G9I0__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\rand_PMC1DC5RA8__01__rand_HPZQ55OXXC__01',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\SAMPLE_DATA',
 'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\sample_data\\wb_dress_sleeveless_H097SG0SIZ__01']

In [11]:
import os
from glob import glob

garment_path_list = glob(os.path.join(
    os.path.abspath(os.getcwd()), "sample_data_deprecated", "*"
))
garment_config_path_list = list(map(
    lambda path : os.path.join(path, f"{os.path.basename(path)}{'' if os.path.basename(path).endswith('__01') else '__01'}__clo.json"),
    garment_path_list 
))

list(map(os.path.exists, garment_config_path_list))

garment_config_path_list

['e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_6W3XGQZOE9\\rand_6W3XGQZOE9__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_828KKFL4TS\\rand_828KKFL4TS__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_8OFBWHKMNI__01__rand_XN21WM295Z__01\\rand_8OFBWHKMNI__01__rand_XN21WM295Z__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_9048W5H2M7__01__rand_AE9AGPLS50__01\\rand_9048W5H2M7__01__rand_AE9AGPLS50__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_GBLADVC34S__01__rand_6AGC4UHTDZ__01\\rand_GBLADVC34S__01__rand_6AGC4UHTDZ__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_KFFMSBTCLN__01__rand_NREVPLXRB4__01\\rand_KFFMSBTCLN__01__rand_NREVPLXRB4__01__clo.json',
 'e:\\HJP\\KUAICV\\VTO\\CLO_AUTO_GEN\\sample_data_deprecated\\rand_LYODEAC7GD__01__rand_NLVEL04PVV__01\\rand_LYODEAC7GD__01__rand_NLVEL04PVV__01__clo.

In [28]:

import os, sys
import json
from pathlib import Path

from glob import glob

with open("gcd_01_top_bottom_path_list.json", "r") as f :
    top_bottom_path_list = json.load(f)

completed_comb_path_list = []
for idx, raw_comb in enumerate(top_bottom_path_list) :
    top_base_path, bottom_base_path = raw_comb.split(",")
    top_garment_split, _, top_garment_id = list(Path(top_base_path).parts)[-3:]
    bottom_garment_split, _, bottom_garment_id = list(Path(bottom_base_path).parts)[-3:]
    
    
    saved_dir = os.path.join(
        r"E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01",
        top_garment_split,
        f"{top_garment_id}__01__{bottom_garment_id}__01"
    )
    
    if len(glob(os.path.join(saved_dir, "*.png"))) > 0 :
        completed_comb_path_list.append(raw_comb)
    
print(len(completed_comb_path_list))

2350


In [4]:
import os, sys
import json
from glob import glob
from pathlib import Path

with open("gcd_01_outfit_path_list.json", "r") as f :
    outfit_path_list = json.load(f)

completed_outfit_path_list = []
for idx, raw_outfit in enumerate(outfit_path_list) :
    base_path = raw_outfit
    
    garment_split, _, garment_id = list(Path(base_path).parts)[-3:]
    
    saved_dir = os.path.join(
        # r"E:\HJP\KUAICV\VTO\DATA\CLO\gcd_01\GCD_01",
        r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN",
        garment_split,
        f"{garment_id}"
    )
    
    if len(glob(os.path.join(saved_dir, "*.png"))) > 0 :
        completed_outfit_path_list.append(raw_outfit)

print(len(completed_outfit_path_list))

0


# Filter out Wrong Fabric

In [ ]:
# import os, sys
# from glob import glob
# import shutil

# from env_constants import SEAMDRESS_SEWFACTORY_DIR

# FABRIC_COUNT = len(glob(os.path.join(
#     r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN",
#     "CLO_ASSETs",
#     "FABRICs",
#     "*.zfab"
# )))

# sewfactory_garment_path_list = sorted(glob(
#     os.path.join(SEAMDRESS_SEWFACTORY_DIR, "sewfactory__01", "*")
# ))[:FABRIC_COUNT]

# for idx, raw_path in enumerate(sewfactory_garment_path_list) :
    
#     target_path = os.path.join(
#         r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN",
#         "SAMPLE_DATA", "SEWFACTORY__01",
#         os.path.basename(raw_path)
#     )
    
#     shutil.copytree(
#         raw_path,
#         target_path,
#         dirs_exist_ok=True
#     )
    

In [ ]:
# import os, sys
# import json
# import time
# from glob import glob
# import random
# import import_api
# import utility_api
# import fabric_api
# import pattern_api
# import export_api
# import ApiTypes

# from pathlib import Path
# from dataclasses import dataclass

# def get_sort_key(path_str):
#     path = Path(path_str)
#     # Find the 'garments_5000_X' part
#     garment_dir = next(part for part in path.parts if part.startswith('garments_5000_'))
#     # Extract the number after 'garments_5000_'
#     garment_num = int(garment_dir.split('_')[-1])
#     # Get the random ID from the path
#     random_id = path.parent.name
#     return (garment_num, random_id)

# @dataclass
# class PathConfig :
#     root_path : str = None
#     avatar_dir: str = "CLO_ASSETS/AVATARs"
#     fabric_dir: str = "CLO_ASSETS/FABRICs"
#     pose_dir  : str = "CLO_ASSETS/POSEs"
#     viewpoint_dir: str = "CLO_ASSETS/VIEWPOINTs"
    
#     gcd_dir: str = "GarmentCodeData_v2"

#     outfit_metadata_path: str = "gcd_01_outfit_path_list.json"
#     combination_metadata_path: str = "gcd_01_top_bottom_path_list.json"
       
#     outfit_path_list: list = None
#     combination_path_list: list = None

#     def __post_init__(self):
#         self.avatar_path_list = sorted(glob(os.path.join(self.root_path, self.avatar_dir, "*.avt")))
#         self.fabric_path_list = sorted(glob(os.path.join(self.root_path, self.fabric_dir, "*.zfab")))
#         self.pose_path_list = sorted(glob(os.path.join(self.root_path, self.pose_dir, "*.pos")))
#         self.viewpoint_path_list = sorted(glob(os.path.join(self.root_path, self.viewpoint_dir, "*.zcmr")))
        

#     @property
#     def avatar_count(self) -> int:
#         return len(self.avatar_path_list)

#     @property
#     def fabric_count(self) -> int:
#         return len(self.fabric_path_list)
        
#     @property
#     def pose_count(self) -> int:
#         return len(self.pose_path_list)
    
#     @property
#     def viewpoint_count(self) -> int:
#         return len(self.viewpoint_path_list)

    
# @dataclass
# class GarmentScene :
#     is_combination: bool = False
#     garment_json_path: str = None
#     avatar_path: str = None
#     whole_fabric_path_list: str = None
#     fabric_path_list: list = None
#     pose_path_list:list    = None
#     panel_count_list: list = None
#     # pose_path:str = None
#     viewpoint_path_list: list = None
    
#     def __post_init__(self):
#         # identify if scene is composed of single or multiple garments
#         if len(os.path.basename(self.garment_json_path).split("__")) == 3:
#             self.is_combination = False
#         elif len(os.path.basename(self.garment_json_path).split("__")) == 5:
#             self.is_combination = True
#         else :
#             raise ValueError(f"Invalid garment json path: {self.garment_json_path}")

#         self.panel_count_list, self.prev_fabric_count = self.get_panel_fabric_count_list()
#         self.fabric_path_list = list(map(
#             lambda x : random.choice(self.whole_fabric_path_list),
#             self.panel_count_list
#         ))
#         # self.fabric_path_list = [self.fabric_path] * len(self.panel_count_list)
#         self.output_dir = os.path.dirname(self.garment_json_path)

#     def get_panel_fabric_count_list(self) :
#         with open(self.garment_json_path, "r") as f:
#             json_data = json.load(f)

#         panel_count_list = []
#         cur_garment_id = None
#         for pattern in json_data["PatternList"] :
#             garment_id = pattern["ID"].split("_")[1]
            
#             if garment_id == cur_garment_id :
#                 panel_count_list[-1] += 1
#             else :
#                 cur_garment_id = garment_id
#                 panel_count_list.append(1)
            
#         return panel_count_list, len(json_data["FabricList"])

#     def import_scene(
#         self, option = None,
#         SIM_STEP = 200,
#         delete_prev_output = False
#     ) :
#         '''
#         import to clo
#         '''
#         if delete_prev_output:
#             output_dir = self.output_dir
#             for file in os.listdir(output_dir):
#                 file_path = os.path.join(output_dir, file)
#                 if not file.endswith('.json') and os.path.isfile(file_path):
#                     try:
#                         os.remove(file_path)
#                     except Exception as e:
#                         print(f"Error deleting {file_path}: {e}")
        
#         utility_api.NewProject()
        
#         if option is None :
#             option = ApiTypes.ImportExportOption()
#             option.bExportGarment = True
#             option.bExportAvatar  = True
#             option.bSingleObject  = True
#             option.bThin          = False
#             option.bSaveInZip     = False
#             option.bMetaData      = True
        
#         import_api.ImportFile(self.avatar_path)
#         pattern_api.ImportPatternJSON(self.garment_json_path)
#         for fabric_path in self.fabric_path_list :
#             fabric_api.AddFabric(fabric_path)
        
#         fabric_idx = fabric_api.GetFabricCount(-2) - len(self.fabric_path_list)
#         colorway_idx = fabric_idx
#         panel_idx = 0
#         for fabric_path, panel_count in zip(self.fabric_path_list, self.panel_count_list) :
#             for _ in range(panel_count) :
                
#                 fabric_api.AssignFabricToPattern(fabric_idx, panel_idx, colorway_idx)
#                 pattern_api.SetArrangementShapeStyle(panel_idx, "Flat")
#                 panel_idx += 1
#             fabric_idx += 1
    
#         export_api.ExportZPac(      os.path.join(self.output_dir, "pre_drape.zpac"))
#         export_api.ExportSnapshot3D(os.path.join(self.output_dir, "pre_drape.png"))
#         import_api.ImportFile(      os.path.join(self.output_dir, "pre_drape.zpac"))
#         utility_api.Simulate(SIM_STEP)
        
#         prev_pose_name = None 
#         for viewpoint in self.viewpoint_path_list :
#             view_name = Path(viewpoint).stem
#             import_api.ImportFile(viewpoint)

#             pose_path = random.choice(self.pose_path_list)
#             pose_name = Path(pose_path).stem
#             import_api.ImportPose(pose_path)
#             # import_api.ImportPose(pose_path)

#             export_api.ExportOBJ(
#                 os.path.join(self.output_dir, f"{pose_name}.obj"),
#                 option
#             )
#             export_api.ExportRenderingImage(os.path.join(
#                 self.output_dir, f"{pose_name}__{view_name}.png"
#             ))
            
#             export_api.ExportOBJ(
#                 os.path.join(self.output_dir, f"{pose_name}.obj"),
#                 option
#             )
#             export_api.ExportRenderingImage(os.path.join(
#                 self.output_dir, f"{pose_name}__{view_name}.png"
#             ))
            
#         # delete every jpg files
#         for file in os.listdir(self.output_dir):
#             if file.endswith(".jpg"):
#                 os.remove(os.path.join(self.output_dir, file))


# SYSTEM_CONFIG_DICT = {
#     "HJP_WINDOWS_DESKTOP": {
#         "CLO_DIR": r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN"
#     }
# }
# system_name = "HJP_WINDOWS_DESKTOP"

# path_config = PathConfig(root_path=SYSTEM_CONFIG_DICT[system_name]["CLO_DIR"])

# GARMENT_VERSION = "03"
# garment_dir_list = sorted(glob(
#     os.path.join(
#         r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN",
#         "SAMPLE_DATA", f"SEWFACTORY__{GARMENT_VERSION}", "*",
#     )
# ))
# garment_path_list = list(map(
#     lambda x : os.path.join(x, f"{os.path.basename(x)}__clo.json"),
#     garment_dir_list
# ))


# for idx, garment_config_json_path in enumerate(garment_path_list) :
#     garment_scene = GarmentScene(
#         garment_json_path=garment_config_json_path,
#         avatar_path=path_config.avatar_path_list[0],
#         whole_fabric_path_list=path_config.fabric_path_list,
#         viewpoint_path_list=path_config.viewpoint_path_list,
#         pose_path_list=path_config.pose_path_list,
#     )
#     garment_scene.import_scene(
#         delete_prev_output=True
#     )

In [ ]:
# import os, sys
# from glob import glob


# gcd_sample_dir_list = sorted(glob(os.path.join(
#     "SAMPLE_DATA", "GCD__01", "*"
# )))

# counter = 0
# for garment_dir in gcd_sample_dir_list :
#     custom_view_path_list = glob(os.path.join(garment_dir, "Custom*.png"))
#     if len(custom_view_path_list) < 10 :
#         counter += 1
        

# print(counter)

262


utility_api

def DeleteAvatar(_avatarIndexList : list[int]) -> bool
"""
@brief Delete Avatars
@param _avatarIndexList: List of indices for avatars to delete
"""

EXAMPLE :
utility_api.DeleteAvatar([0])


In [ ]:
# garment_path_list = sorted(glob(
#     os.path.join(
#         r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN",
#         "SAMPLE_DATA", "SEWFACTORY__02", "*__02",
#         "*clo.json"
#     )
# ))
# len(garment_path_list)

363

In [26]:
import json


BASE_NAME = os.path.basename("D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY\dress_sleeveless_005CWF73E9\dress_sleeveless_005CWF73E9_specification.json")

BASE_NAME.split("__")





['dress_sleeveless_005CWF73E9_specification.json']

In [28]:
PP = r"\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY\dress_sleeveless_005CWF73E9__01\dress_sleeveless_005CWF73E9_specification.json"

os.path.exists(PP)

True

In [ ]:
import import_api
import export_api
import utility_api
import fabric_api
import pattern_api
import ApiTypes


P1 = r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY__03\tee_sleeveless_15PNW7FMCZ__03__skirt_4_panels_ZQ5C161PAR__03\skirt_4_panels_ZQ5C161PAR__03__clo.zprj"
P1 = r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY__03\tee_sleeveless_177RQTNAWB__03__skirt_8_panels_UVL9431GP1__03\skirt_8_panels_UVL9431GP1__03__clo.zprj"

P2 = r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY__03\tee_sleeveless_15PNW7FMCZ__03__skirt_4_panels_ZQ5C161PAR__03\tee_sleeveless_15PNW7FMCZ__03__clo.zprj"
P2 = r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY__03\tee_sleeveless_177RQTNAWB__03__skirt_8_panels_UVL9431GP1__03\tee_sleeveless_177RQTNAWB__03__clo.zprj"

SIM_STEP = 100

zprj_option = ApiTypes.ImportZPRJOption()
zprj_option.bAppend = True

utility_api.NewProject()
import_api.ImportZprj(P1, zprj_option)
utility_api.Simulate(SIM_STEP)

import_api.ImportZprj(P2, zprj_option)
utility_api.Simulate(SIM_STEP)








In [42]:
os.path.exists(
    r"D:\VTO2025\DATASETs\Ours1\CLO_AUTO_GEN\SAMPLE_DATA\SEWFACTORY__03\tee_sleeveless_15PNW7FMCZ__03__skirt_4_panels_ZQ5C161PAR__03\tee_sleeveless_15PNW7FMCZ__03__clo.zprj"
)

True

In [39]:
GARMENT_VERSION = "03"

for dir in list(filter(
    lambda x : x[-4:] != f"__{GARMENT_VERSION}",
    sorted(glob(os.path.join(
        "SAMPLE_DATA",
        f"SEWFACTORY__{GARMENT_VERSION}",
        "*"
    )))
)) :
    # remove tree
    shutil.rmtree(dir)

In [44]:
PP = r'D:\\VTO2025\\DATASETs\\Ours1\\CLO_AUTO_GEN\\SAMPLE_DATA\\SEWFACTORY__03\\dress_sleeveless_005CWF73E9\\dress_sleeveless_005CWF73E9__dress_sleeveless_005CWF73E9__clo.json'

os.path.exists(PP)

False